# **Importing the required libraries**

In [ ]:
import os, random, shutil, numpy as np, tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications.resnet50 import preprocess_input
from google.colab import drive

In [ ]:
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
np.random.seed(SEED)
tf.random.set_seed(SEED)

class_weight = {
    0: 2.0,  # AD
    1: 1.0   # CN
}

# **Data Loading as in  the  SimCLR_MultiView_FeatureLearning.ipynb notebook**

In [ ]:
drive.mount('/content/drive', force_remount=True)

LOCAL_PATH = "/content/ADNI_MultiView_Dataset"
if not os.path.exists(LOCAL_PATH):
    shutil.copytree("/content/drive/MyDrive/ADNI_MultiView_Dataset", LOCAL_PATH)


IMG_SIZE = 224
BATCH_SIZE_FUSION = 16

In [ ]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    return tf.image.resize(img, (IMG_SIZE, IMG_SIZE))

def load_multiview_dataset(path):

    target_classes = ["AD", "CN"]
    data = []
    for label, cls in enumerate(target_classes):
        cls_path = os.path.join(path, cls)
        if not os.path.exists(cls_path): continue
        patient_dict = {}
        for file in os.listdir(cls_path):
            if not file.lower().endswith(".png"): continue
            pid = file.split("_brain_norm_")[0]
            if pid not in patient_dict: patient_dict[pid] = {}
            for v in ["axial", "sag", "cor"]:
                if v in file.lower(): patient_dict[pid][v] = os.path.join(cls_path, file)
        for pid, views in patient_dict.items():
            if len(views) == 3: data.append((views["axial"], views["sag"], views["cor"], label))
    return data, target_classes

def process_fusion(axial, sag, cor, label):
    def prep(p): return preprocess_input(load_image(p))
    return {"axial_input": prep(axial), "sagittal_input": prep(sag), "coronal_input": prep(cor)}, label

def build_fusion_ds(data, training=True):
    ax, sg, cr = [tf.constant([x[i] for x in data], dtype=tf.string) for i in range(3)]
    lbls = tf.constant([x[3] for x in data], dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((ax, sg, cr, lbls)).map(process_fusion)
    if training: ds = ds.shuffle(1000).repeat()
    return ds.batch(BATCH_SIZE_FUSION).prefetch(tf.data.AUTOTUNE)

In [ ]:
train_data, CLASS_NAMES = load_multiview_dataset(os.path.join(LOCAL_PATH, "train"))
val_data, _ = load_multiview_dataset(os.path.join(LOCAL_PATH, "val"))
test_data, _ = load_multiview_dataset(os.path.join(LOCAL_PATH, "test"))

train_ds = build_fusion_ds(train_data, training=True)
val_ds = build_fusion_ds(val_data, training=False)

# **Loading the Pre-trained SimCLR encoder from the SimCLR_MultiView_FeatureLearning.ipynb notebook**

In [ ]:
from tensorflow.keras.models import load_model

# Load previously trained SimCLR + fusion model
saved_path = '/content/drive/MyDrive/ADNI_SimCLR_MultiView_Final.h5'
full_saved_model = load_model(saved_path)

# Extract encoder (feature extractor)
encoder = full_saved_model.get_layer("Encoder")

# Freeze encoder to preserve learned representations
encoder.trainable = False

# **Custom SOM Layer**

In [ ]:
class SOMLayer(layers.Layer):
    """
    Self-Organizing Map layer with neighbourhood regularisation.

    Computes Euclidean distances from the input to every prototype, returns
    those distances to the classification head, and simultaneously adds a
    soft neighbourhood loss that encourages topological ordering.
    """
    def __init__(self, map_size=(10, 10), som_lambda=0.01, **kwargs):
        super(SOMLayer, self).__init__(**kwargs)
        self.map_size   = map_size
        self.n_nodes    = map_size[0] * map_size[1]
        self.som_lambda = som_lambda

    def build(self, input_shape):
        self.prototypes = self.add_weight(
            name='prototypes',
            shape=(self.n_nodes, input_shape[-1]),
            initializer=tf.keras.initializers.GlorotUniform(seed=42),
            trainable=True
        )
        super(SOMLayer, self).build(input_shape)

    def call(self, inputs):
        diff      = tf.expand_dims(inputs, axis=1) - self.prototypes
        distances = tf.math.sqrt(tf.reduce_sum(tf.square(diff), axis=-1) + 1e-8)

        # Neighbourhood loss: pull input toward its closest prototype
        bmu_idx        = tf.argmin(distances, axis=1)
        bmu_protos     = tf.gather(self.prototypes, bmu_idx)
        neighbour_loss = tf.reduce_mean(
            tf.reduce_sum(tf.square(inputs - bmu_protos), axis=-1)
        )
        self.add_loss(self.som_lambda * neighbour_loss)

        return distances

# **Hybrid Model Architechture (CNN-SOM)**

In [ ]:
in_a, in_s, in_c = [layers.Input(shape=(224,224,3), name=n) for n in ["axial_input", "sagittal_input", "coronal_input"]]
merged = layers.Concatenate()([encoder(in_a), encoder(in_s), encoder(in_c)])
x = layers.Dense(512, activation='relu')(merged)
x = layers.Dropout(0.3)(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.BatchNormalization()(x)

som_dists = SOMLayer(map_size=(10, 10), name="SOM_Layer")(x)
outputs   = layers.Dense(len(CLASS_NAMES), activation='softmax')(som_dists)

hybrid_som_model = Model(inputs=[in_a, in_s, in_c], outputs=outputs)

In [ ]:
#Compile
hybrid_som_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
    )

In [ ]:
MODEL_SAVE_PATH = '/content/drive/MyDrive/hybrid_som_model_final_1.h5'

if os.path.exists(MODEL_SAVE_PATH):
    hybrid_som_model = load_model(
        MODEL_SAVE_PATH,
        custom_objects={"SOMLayer": SOMLayer}
    )
    # Re-resolve BN layer name after loading
    BN_LAYER_NAME = [
        l.name for l in hybrid_som_model.layers
        if isinstance(l, layers.BatchNormalization)
    ][-1]
    print("Model loaded.")
else:
    hybrid_som_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        steps_per_epoch=len(train_data) // BATCH_SIZE_FUSION,
        class_weight=class_weight
    )
    hybrid_som_model.save(MODEL_SAVE_PATH)
    print("Model saved.")

In [ ]:
import os

# Folder for visual results
SAVE_DIR = '/content/drive/MyDrive/Thesis_Results_Final_One'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

def save_thesis_plot(filename):
    path = os.path.join(SAVE_DIR, filename)
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print(f"Saved: {path}")

# **Evaluation**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Re-evaluate y_true and y_pred if not defined (e.g., after kernel restart or out-of-order execution)
if 'y_true' not in locals() or 'y_true' not in globals():
    print("Recomputing y_true and y_pred for evaluation...")
    test_ds = build_fusion_ds(test_data, training=False)

    y_true_temp, y_pred_temp = [], []

    for inputs, labels in test_ds:
        preds = hybrid_som_model.predict(inputs, verbose=0)
        y_true_temp.extend(labels.numpy())
        y_pred_temp.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true_temp)
    y_pred = np.array(y_pred_temp)

# Classification Report
print("\n HYBRID CNN-SOM REPORT")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix
plt.figure(figsize=(6,5))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
save_thesis_plot('Figure_4_Confusion_Matrix.png')
plt.show()

In [ ]:
## Threshold Tuning — Optimised for AD Recall

threshold = 0.1

# Ensure y_prob, y_true, y_pred are defined for threshold tuning
if 'y_prob' not in locals() or 'y_prob' not in globals():
    print("Recomputing y_true, y_pred, and y_prob for threshold tuning...")
    test_ds = build_fusion_ds(test_data, training=False) # Ensure test_ds is available

    y_true_temp, y_pred_temp, y_prob_temp = [], [], []

    for inputs, labels in test_ds:
        preds = hybrid_som_model.predict(inputs, verbose=0)
        y_true_temp.extend(labels.numpy())
        y_pred_temp.extend(np.argmax(preds, axis=1))
        y_prob_temp.extend(preds[:, 0]) # AD probability for ROC

    y_true = np.array(y_true_temp)
    y_pred = np.array(y_pred_temp)
    y_prob = np.array(y_prob_temp)

y_pred_tuned = np.where(y_prob >= threshold, 0, 1)

print(f"\n TUNED REPORT OF THE HYBRID CNN-SOM MODEL (threshold = {threshold})")
print(classification_report(y_true, y_pred_tuned, target_names=CLASS_NAMES))

# Tuned Confusion Matrix
plt.figure(figsize=(6,5))
sns.heatmap(confusion_matrix(y_true, y_pred_tuned), annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f"Confusion Matrix (Threshold = {threshold})")
save_thesis_plot('Figure_6_Confusion_Matrix_Tuned.png')
plt.show()

In [ ]:
from sklearn.metrics import  roc_curve, auc

test_ds = build_fusion_ds(test_data, training=False)

y_true, y_pred, y_prob = [], [], []

for inputs, labels in test_ds:
    preds = hybrid_som_model.predict(inputs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_prob.extend(preds[:, 0])          # AD probability for ROC

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_prob, pos_label=0)
roc_auc     = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0,1],[0,1], color='navy', lw=1.5, linestyle='--', label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curve — Hybrid CNN-SOM (AD vs CN)', fontsize=15, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
save_thesis_plot('Figure_5_ROC_Curve.png')
plt.show()
print(f"\n AUC Score: {roc_auc:.4f}")

# **Visualization**

##U-MATRIX heatmap (SOM TOPOLOGY)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def generate_som_viz(model):
    # Extract and reshape prototype weights to (10, 10, feature_dim)
    weights = model.get_layer("SOM_Layer").get_weights()[0]
    map_size = (10, 10)
    grid = weights.reshape(map_size[0], map_size[1], -1)

    #Compute mean distance to cardinal neighbours for each node
    u_matrix = np.zeros(map_size)
    for y in range(map_size[0]):
        for x in range(map_size[1]):
            dists = []
            if x > 0: dists.append(np.linalg.norm(grid[y,x] - grid[y, x-1]))
            if x < 9: dists.append(np.linalg.norm(grid[y,x] - grid[y, x+1]))
            if y > 0: dists.append(np.linalg.norm(grid[y,x] - grid[y-1, x]))
            if y < 9: dists.append(np.linalg.norm(grid[y,x] - grid[y+1, x]))
            u_matrix[y,x] = np.mean(dists)

    # U-matrix Plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(u_matrix, cmap='magma', annot=True, fmt=".2f")
    plt.title("U-Matrix: Inherent Topological Clusters of Brain Atrophy")
    plt.xlabel("SOM X-Dimension")
    plt.ylabel("SOM Y-Dimension")
    save_thesis_plot('Figure_1_UMatrix_Heatmap.png')
    plt.show()

generate_som_viz(hybrid_som_model)

## Class Territory Map

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_class_territories(model, dataset):
    # Determine the name of the last BatchNormalization layer dynamically
    # This is needed because BN_LAYER_NAME might not be globally defined if the model was trained in this run
    bn_layer_name = [
        l.name for l in model.layers
        if isinstance(l, layers.BatchNormalization)
    ][-1]

    # Get the feature extractor output before the SOM layer..
    feature_extractor = Model(inputs=model.input, outputs=model.get_layer(bn_layer_name).output)

    som_weights = model.get_layer("SOM_Layer").get_weights()[0]
    class_map = {0: [], 1: []} # 0 for AD, 1 for CN

    print("Mapping test patients to SOM grid...")
    for inputs, labels in dataset:
        feats = feature_extractor.predict(inputs, verbose=0)

        for i, f in enumerate(feats):
            # Find Best Matching Unit (BMU)
            dists = np.linalg.norm(som_weights - f, axis=1)
            bmu = np.argmin(dists)

            y, x = divmod(bmu, 10)
            class_map[labels[i].numpy()].append((x, y))

    # Plotting
    plt.figure(figsize=(10, 10))

    ad_pts = np.array(class_map[0])
    cn_pts = np.array(class_map[1])

    if len(ad_pts) > 0:
        plt.scatter(ad_pts[:, 0], ad_pts[:, 1], c='red', label='AD (Alzheimer)',
                    alpha=0.7, marker='x', s=100, linewidths=2)
    if len(cn_pts) > 0:
        plt.scatter(cn_pts[:, 0], cn_pts[:, 1], c='blue', label='CN (Healthy)',
                    alpha=0.7, marker='o', s=100)

    plt.xticks(range(10))
    plt.yticks(range(10))
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.title("Inherent Interpretability: Class Territories on SOM Grid")
    plt.xlabel("SOM Dimension 1")
    plt.ylabel("SOM Dimension 2")
    save_thesis_plot('Figure_2_SOM_Territory_Map.png')
    plt.legend()
    plt.show()

test_ds = build_fusion_ds(test_data, training=False)
plot_class_territories(hybrid_som_model, test_ds)

## GRAD-CAM: Local Saliency (Where the model looks)

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="conv5_block3_out"):

    encoder = model.get_layer("Encoder")
    grad_model = tf.keras.models.Model(
        [encoder.inputs], [encoder.get_layer(last_conv_layer_name).output, encoder.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        class_channel = preds[:, 0] # Focus on AD class

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap))
    return heatmap.numpy()

def display_gradcam(img, heatmap, alpha=0.4):
    # Get original image dimensions
    # img is the raw numpy array from Gradio (H, W, C)
    height, width = img.shape[0], img.shape[1]

    # Rescale heatmap to 0-255
    heatmap = np.uint8(255 * heatmap)

    # Apply the jet colormap
    jet = plt.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]

    # Resize heatmap to match the ORIGINAL image dimensions
    jet_heatmap = tf.keras.utils.array_to_img(jet_heatmap).resize((width, height))
    jet_heatmap = tf.keras.utils.img_to_array(jet_heatmap)

    # Blend the images
    superimposed_img = jet_heatmap * alpha + img
    return tf.keras.utils.array_to_img(superimposed_img)

## Build SOM Node ATLAS (BMU + Training Image Lookup Table)

In [ ]:
feature_extractor = Model(inputs=hybrid_som_model.input,
                          outputs=hybrid_som_model.get_layer(BN_LAYER_NAME).output)
som_weights = hybrid_som_model.get_layer("SOM_Layer").get_weights()[0]

import pickle
ATLAS_SAVE_PATH = '/content/drive/MyDrive/node_to_image_paths.pkl'

if os.path.exists(ATLAS_SAVE_PATH):
    with open(ATLAS_SAVE_PATH, 'rb') as f:
        node_to_image_paths = pickle.load(f)
    print("Atlas loaded.")
else:
    node_to_image_paths = {i: [] for i in range(100)}
    for i in range(len(train_data)):
        axial, sag, cor, label = train_data[i]
        inputs, _ = process_fusion(axial, sag, cor, label)
        inputs    = {k: tf.expand_dims(v, 0) for k, v in inputs.items()}
        feats     = feature_extractor.predict(inputs, verbose=0)
        dists     = np.linalg.norm(som_weights - feats, axis=1)
        bmu       = np.argmin(dists)
        node_to_image_paths[bmu].append(axial)
    with open(ATLAS_SAVE_PATH, 'wb') as f:
        pickle.dump(node_to_image_paths, f)
    print("Atlas saved.")

## Case-Based Explanation (Human Explanation)

In [ ]:
import matplotlib.pyplot as plt

# ── Rebuild node_to_image_paths fresh (ensures consistency with current weights) ──
from collections import defaultdict

node_to_image_paths = defaultdict(list)
for axial, sag, cor, label in train_data:
    inputs, _ = process_fusion(axial, sag, cor, label)
    inputs = {k: tf.expand_dims(v, 0) for k, v in inputs.items()}
    feats = feature_extractor.predict(inputs, verbose=0)
    bmu = int(np.argmin(np.linalg.norm(som_weights - feats, axis=1)))
    node_to_image_paths[bmu].append({"axial": axial, "sagittal": sag, "coronal": cor})

print(f"Rebuilt. {len(node_to_image_paths)} nodes populated.")

# ── Find a good patient index (maps to a node with ≥3 evidence images) ────────
good_index = None
for i, (axial, sag, cor, label) in enumerate(test_data):
    inputs, _ = process_fusion(axial, sag, cor, label)
    inputs = {k: tf.expand_dims(v, 0) for k, v in inputs.items()}
    feats = feature_extractor.predict(inputs, verbose=0)
    bmu = int(np.argmin(np.linalg.norm(som_weights - feats, axis=1)))
    if len(node_to_image_paths.get(bmu, [])) >= 3:
        pred = int(np.argmax(hybrid_som_model.predict(inputs, verbose=0)))
        if CLASS_NAMES[pred] == "AD":   # prefer a correctly-interesting AD case
            good_index = i
            break

if good_index is None:  # fallback: any index with ≥3 evidence
    for i, (axial, sag, cor, label) in enumerate(test_data):
        inputs, _ = process_fusion(axial, sag, cor, label)
        inputs = {k: tf.expand_dims(v, 0) for k, v in inputs.items()}
        feats = feature_extractor.predict(inputs, verbose=0)
        bmu = int(np.argmin(np.linalg.norm(som_weights - feats, axis=1)))
        if len(node_to_image_paths.get(bmu, [])) >= 3:
            good_index = i
            break

print(f"Using patient index: {good_index}")


def explain_like_a_doctor(patient_index, test_data_list, node_lookup):
    axial, sag, cor, true_label = test_data_list[patient_index]
    inputs, _ = process_fusion(axial, sag, cor, true_label)
    inputs = {k: tf.expand_dims(v, 0) for k, v in inputs.items()}

    preds = hybrid_som_model.predict(inputs, verbose=0)
    pred_label = np.argmax(preds)

    feats = feature_extractor.predict(inputs, verbose=0)
    dists = np.linalg.norm(som_weights - feats, axis=1)
    bmu = int(np.argmin(dists))  # ← cast to int so it works as a dict key

    evidence_paths = node_lookup[bmu][:3]

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(plt.imread(axial), cmap='gray')
    plt.title(f"TEST PATIENT\nPred: {CLASS_NAMES[pred_label]}")
    plt.axis('off')

    for i, path in enumerate(evidence_paths):
        plt.subplot(1, 4, i + 2)
        plt.imshow(plt.imread(path["axial"]), cmap='gray') # Fixed: Access 'axial' key from the dictionary
        plt.title(f"Evidence Case {i+1}\n(Node #{bmu})")
        plt.axis('off')

    plt.suptitle(f"INHERENT EXPLANATION: Patient mapped to Topological Node {bmu}", fontsize=16)
    save_thesis_plot('Figure_3_Case_Based_Explanation.png')
    plt.show()

explain_like_a_doctor(good_index, test_data, node_to_image_paths)


In [ ]:
print(f"Node 12 has {len(node_to_image_paths.get(12, []))} images")
print(f"Total nodes with images: {len([k for k, v in node_to_image_paths.items() if len(v) > 0])}")
print(f"Nodes with images: {sorted([k for k, v in node_to_image_paths.items() if len(v) > 0])}")

## Interpretability Comparison (LIME vs SOM)

In [ ]:
## Interpretability Comparison (LIME vs SOM)

!pip install lime shap -q
import lime
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
import numpy as np

def generate_comparison_report(patient_index, test_data_list):

    # ── 1. Setup Patient Data ─────────────────────────────────────────────
    axial_path, sag_path, cor_path, label = test_data_list[patient_index]

    # Load raw image for LIME (must be 0–255, no preprocessing yet)
    axial_img_raw = plt.imread(axial_path)

    if axial_img_raw.ndim == 2:
        axial_img_raw = np.stack([axial_img_raw]*3, axis=-1)
    elif axial_img_raw.ndim == 3 and axial_img_raw.shape[-1] == 4:
        axial_img_raw = axial_img_raw[:, :, :3]

    if axial_img_raw.max() <= 1.0:
        axial_img_raw = (axial_img_raw * 255).astype(np.uint8)

    # ── 2. Preprocess fixed sagittal and coronal views ONCE ───────────────
    def preprocess(img_array):
        if img_array.ndim == 2:
            img_array = np.stack([img_array]*3, axis=-1)
        elif img_array.ndim == 3 and img_array.shape[-1] == 4:
            img_array = img_array[:, :, :3]
        img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)
        img_resized = tf.image.resize(img_tensor, (224, 224))
        return preprocess_input(img_resized)

    in_s = tf.expand_dims(preprocess(plt.imread(sag_path)), 0)
    in_c = tf.expand_dims(preprocess(plt.imread(cor_path)), 0)

    # ── 3. LIME predict_fn — only axial perturbed, others held fixed ──────
    def predict_fn(images):
        # Apply preprocessing (including resizing) to LIME's perturbed images
        processed_images = []
        for img in images:
            processed_images.append(preprocess(img)) # Use the local preprocess function
        processed_images = tf.stack(processed_images)

        sag_batch = tf.repeat(in_s, repeats=len(images), axis=0)
        cor_batch = tf.repeat(in_c, repeats=len(images), axis=0)
        return hybrid_som_model.predict([processed_images, sag_batch, cor_batch], verbose=0)

    # ── 4. LIME Explanation ───────────────────────────────────────────────
    explainer = lime_image.LimeImageExplainer()

    explanation = explainer.explain_instance(
        axial_img_raw.astype('double'),
        predict_fn,
        top_labels=1,
        hide_color=0,
        num_samples=1000
    )

    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0],
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    # ── 5. SOM Prototype ──────────────────────────────────────────────────
    in_a = tf.expand_dims(preprocess(plt.imread(axial_path)), 0)

    dist_model = Model(
        inputs=hybrid_som_model.input,
        outputs=hybrid_som_model.get_layer("SOM_Layer").output
    )
    dists = dist_model.predict([in_a, in_s, in_c], verbose=0)
    bmu   = np.argmin(dists[0])

    # Handle cases where the BMU might not have any associated images
    if node_to_image_paths[bmu]:
        evidence = plt.imread(node_to_image_paths[bmu][0]["axial"]) # Fixed: Access 'axial' key
    else:
        # Node is empty — find the nearest non-empty node
        print(f"Node {bmu} is empty, searching nearest non-empty node...")
        non_empty_nodes = [i for i in range(100) if node_to_image_paths[i]]
        if non_empty_nodes:
            # Get SOM weights and find closest populated node
            som_w = hybrid_som_model.get_layer("SOM_Layer").get_weights()[0]
            bmu_vec = som_w[bmu]
            fallback_bmu = min(
                non_empty_nodes,
                key=lambda i: np.linalg.norm(som_w[i] - bmu_vec)
            )
            evidence = plt.imread(node_to_image_paths[fallback_bmu][0]["axial"]) # Fixed: Access 'axial' key
            print(f"Using nearest non-empty node: {fallback_bmu}")
        else:
            evidence = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    # ── 6. Visualization ──────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(axial_img_raw / 255.0)
    axes[0].set_title(f"Original Scan\n(True Label: {CLASS_NAMES[label]})", fontsize=13)
    axes[0].axis('off')

    lime_display = temp / temp.max() if temp.max() > 0 else temp
    axes[1].imshow(mark_boundaries(lime_display, mask))
    axes[1].set_title("LIME Explanation\n(Post-hoc, Axial View)", fontsize=13)
    axes[1].axis('off')

    axes[2].imshow(evidence, cmap='gray')
    axes[2].set_title(f"SOM Prototype (Node {bmu})\n(Inherent, Clinical Evidence)", fontsize=13)
    axes[2].axis('off')

    plt.suptitle("Post-hoc (LIME) vs. Inherent (SOM) Interpretability",
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_thesis_plot(f'Comparison_Patient_{patient_index}.png')
    plt.show()


generate_comparison_report(10, test_data)

#GUI

In [ ]:
import gradio as gr

def predict_explain_ultimate(axial_img, sag_img, cor_img):
    try:
        # Preprocess
        def p(img):
            return preprocess_input(tf.image.resize(tf.convert_to_tensor(img, dtype=tf.float32), (224,224)))

        in_a = tf.expand_dims(p(axial_img), 0)
        in_s = tf.expand_dims(p(sag_img), 0)
        in_c = tf.expand_dims(p(cor_img), 0)

        # Get Prediction
        preds = hybrid_som_model.predict([in_a, in_s, in_c], verbose=0)
        prob_ad = float(preds[0][0])
        label_str = "AD" if prob_ad > 0.5 else "CN"

        # Generate Grad-CAM (The 'Where')
        # Using the Axial view as the primary saliency reference
        heatmap = make_gradcam_heatmap(in_a, hybrid_som_model)
        cam_result = display_gradcam(axial_img, heatmap)

        # Get SOM Evidence (The 'What')
        dist_model = Model(inputs=hybrid_som_model.input, outputs=hybrid_som_model.get_layer("SOM_Layer").output)
        dists = dist_model.predict([in_a, in_s, in_c], verbose=0)
        bmu = np.argmin(dists[0])

        evidence_list = []
        if bmu in node_to_image_paths and node_to_image_paths[bmu]:
            paths = node_to_image_paths[bmu][:3]
            for path in paths:
                evidence_list.append(plt.imread(path))

        while len(evidence_list) < 3:
            evidence_list.append(np.zeros((224,224,3)))

        return (
            {"Alzheimer (AD)": prob_ad, "Healthy (CN)": 1-prob_ad},
            cam_result, # The Heatmap
            f"Node #{bmu}: Atrophy Pattern Identified",
            evidence_list[0], evidence_list[1], evidence_list[2]
        )
    except Exception as e:
        return None, None, f"Error: {str(e)}", None, None, None


with gr.Blocks(css=".gradio-container {background-color: #121212; color: #ff8c00;}") as demo:
    gr.Markdown("<h1 style='color: #ff8c00; text-align: center;'>Hybrid CNN-SOM Clinical Dashboard</h1>")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("<h4 style='color: #ff8c00;'>1. Patient Input</h4>")
            a, s, c = gr.Image(label="Axial"), gr.Image(label="Sagittal"), gr.Image(label="Coronal")
            go = gr.Button("RUN DIAGNOSTICS ", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("<h4 style='color: #ff8c00;'>2. Local Saliency (Grad-CAM)</h4>")
            # This shows WHERE the model is looking
            out_cam = gr.Image(label="Focus Regions (Heatmap)")

            gr.Markdown("<h4 style='color: #ff8c00;'>3. Prediction & Reasoning</h4>")
            out_label = gr.Label(num_top_classes=2)
            out_txt = gr.Textbox(label="Topological Mapping")

            gr.Markdown("<h4 style='color: #ff8c00;'>4. Global Evidence (SOM Prototypes)</h4>")
            # This shows WHAT the model is comparing it to
            with gr.Row():
                ev1 = gr.Image(label="Ref 1")
                ev2 = gr.Image(label="Ref 2")
                ev3 = gr.Image(label="Ref 3")

    go.click(predict_explain_ultimate, [a, s, c], [out_label, out_cam, out_txt, ev1, ev2, ev3])

demo.launch(share=True)

In [ ]:
import pickle, os, shutil
from huggingface_hub import HfApi, login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

REPO_ID = "Chen-T/hybrid-cnn-som-ad-classification"
api      = HfApi()


PROTO_DIR = '/content/prototype_images'
os.makedirs(PROTO_DIR, exist_ok=True)

node_to_proto = {}
for node_id, cases in node_to_image_paths.items():
    if cases:
        case = cases[0]
        node_to_proto[node_id] = {}
        for view in ["axial", "sagittal", "coronal"]:
            fname = f"node_{node_id}_{view}.png"
            dest = os.path.join(PROTO_DIR, fname)
            shutil.copy(case[view], dest)
            node_to_proto[node_id][view] = f"prototype_images/{fname}"

print(f"{len(node_to_proto)} nodes ready ({len(node_to_proto)*3} images total).")

with open('/content/node_to_proto.pkl', 'wb') as f:
    pickle.dump(node_to_proto, f)
print("Atlas saved.")

api.upload_file(
    path_or_fileobj='/content/node_to_proto.pkl',
    path_in_repo='node_to_proto.pkl',
    repo_id=REPO_ID,
    repo_type='space'
)
print("Atlas uploaded.")

print("Uploading prototype images... this may take a minute.")
for filename in os.listdir(PROTO_DIR):
    if filename.endswith('.png'):
        api.upload_file(
            path_or_fileobj=os.path.join(PROTO_DIR, filename),
            path_in_repo=f"prototype_images/{filename}",
            repo_id=REPO_ID,
            repo_type='space'
        )
print("All images uploaded!")
print(f"\n Done! Check your Space files at:")
print(f"   https://huggingface.co/spaces/{REPO_ID}")